In [1]:
# =========================================================
# 1. Imports and project paths
# =========================================================

# Import garbage collection so completed model objects can be released.
import gc

# Import sys so the project root can be added to Python's module path.
import sys

# Import os so the feature set and search mode can be selected from the terminal.
import os

# Import Path for Linux/WSL-safe file paths.
from pathlib import Path

# Import NumPy for arrays, target transformation, indexing, and averages.
import numpy as np

# Import pandas for parquet input and CSV results.
import pandas as pd

# Import CatBoostRegressor for the two independent target models.
from catboost import CatBoostRegressor

# Import the CatBoost GPU-count utility for a clear startup check.
from catboost.utils import get_gpu_device_count

# Import PCA so MiniLM embeddings can be reduced inside each fold.
from sklearn.decomposition import PCA

# Import ParameterGrid for the complete exhaustive Experiment 2 search.
from sklearn.model_selection import ParameterGrid

# Import ParameterSampler for the reduced random search.
from sklearn.model_selection import ParameterSampler

# Define the root of the bikeshare project inside WSL.
PROJECT_ROOT = Path(
    "/home/najla/dev/najla-msc/bikeshare"
)

In [2]:
path = "/home/najla/dev/najla-msc/bikeshare/data/processed/graph/all_wikidata_tokens.parquet"
df = pd.read_parquet(path)

df.head()

,loc_id,attraction_count,wiki_items_text
0,5350001.00,13,artificial_island bay beach building bus_garag...
1,5350002.00,23,airport commercial_traffic_aerodrome airport_...
2,5350003.00,7,changing_room pavilion filling_station monumen...
3,5350004.00,4,destroyed_building_or_structure movie_theater ...
4,5350005.00,10,Catholic_seminary Little_Tibet neighborhood ca...


In [4]:
# Split the whitespace-separated Wikidata tokens into one token per row.
# Keep only loc_id and the individual token; drop attraction_count.
melted_df = (
    df[["loc_id", "wiki_items_text"]]
    .assign(wikidata_token=lambda x: x["wiki_items_text"].str.split())
    .explode("wikidata_token", ignore_index=True)
    .drop(columns=["wiki_items_text"])
)

melted_df

,loc_id,wikidata_token
0,5350001.00,artificial_island
1,5350001.00,bay
2,5350001.00,beach
3,5350001.00,building
4,5350001.00,bus_garage
...,...,...
2702,5350802.02,neighborhood
2703,5350802.02,urban-type_settlement
2704,5350803.03,federal_electoral_district_in_Ontario
2705,5350803.03,federal_electoral_district_of_Canada


In [5]:
# Write the melted token table to a new parquet file.
output_path = "/home/najla/dev/najla-msc/bikeshare/data/processed/graph/melted_wikidata_tokens.parquet"
melted_df.to_parquet(output_path, index=False)

# Preview the saved result.
melted_df.head()

,loc_id,wikidata_token
0,5350001.00,artificial_island
1,5350001.00,bay
2,5350001.00,beach
3,5350001.00,building
4,5350001.00,bus_garage


In [5]:
# =========================================================
# 1. Read the categorised Wikidata token CSV
# =========================================================

from pathlib import Path
import pandas as pd

graph_dir = Path("/home/najla/dev/najla-msc/bikeshare/data/processed/graph")

categorised_path = graph_dir / "melted_wikidata_categorised.csv"
df = pd.read_csv(categorised_path)

df.head()

,loc_id,wikidata_token,minilm_cluster,final_category
0,5350001.0,artificial_island,0,"Parks, Nature & Recreation"
1,5350001.0,bay,0,"Parks, Nature & Recreation"
2,5350001.0,beach,0,"Parks, Nature & Recreation"
3,5350001.0,building,17,Generic Buildings & Structures
4,5350001.0,bus_garage,11,Transport & Transit


In [6]:
# =========================================================
# 2. Keep only loc_id and final category
# =========================================================

# Keep the location id and final semantic category only.
# Rename final_category to category so the cleaning code is shorter and clearer.
df = df[["loc_id", "final_category"]].rename(
    columns={"final_category": "category"}
)

# Convert loc_id to a consistent two-decimal string for reliable joins.
df["loc_id"] = (
    pd.to_numeric(df["loc_id"], errors="coerce")
    .map(lambda value: f"{value:.2f}" if pd.notna(value) else pd.NA)
    .astype("string")
)

# Preview the reduced table.
df.head()

,loc_id,category
0,5350001.00,"Parks, Nature & Recreation"
1,5350001.00,"Parks, Nature & Recreation"
2,5350001.00,"Parks, Nature & Recreation"
3,5350001.00,Generic Buildings & Structures
4,5350001.00,Transport & Transit


In [7]:
# =========================================================
# 3. Clean category text
# =========================================================

# Standardise category text by:
# 1. converting to string
# 2. trimming extra spaces
# 3. converting all text to lower case
# 4. replacing "&" with "and"
# 5. removing special characters such as commas
# 6. replacing spaces with underscores so each category becomes one token
df["category"] = (
    df["category"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace("&", " and ", regex=False)
    .str.replace("_", " ", regex=False)
    .str.replace(r"[^a-z0-9\s]+", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.replace(" ", "_", regex=False)
)

# Preview the cleaned category tokens.
df.head()

,loc_id,category
0,5350001.00,parks_nature_and_recreation
1,5350001.00,parks_nature_and_recreation
2,5350001.00,parks_nature_and_recreation
3,5350001.00,generic_buildings_and_structures
4,5350001.00,transport_and_transit


In [8]:
# =========================================================
# 4. Aggregate different categories by loc_id
# =========================================================

# For each location, keep each different category once.
# Join the unique categories with spaces to create one text field per loc_id.
cat_wikidata_tokens = (
    df.dropna(subset=["category"])
    .drop_duplicates(subset=["loc_id", "category"])
    .groupby("loc_id", sort=False, as_index=False)["category"]
    .agg(lambda values: " ".join(values))
)

# Preview the aggregated category text.
cat_wikidata_tokens.head()

,loc_id,category
0,5350001.00,parks_nature_and_recreation generic_buildings_...
1,5350002.00,transport_and_transit parks_nature_and_recreat...
2,5350003.00,generic_buildings_and_structures retail_and_co...
3,5350004.00,generic_buildings_and_structures culture_and_p...
4,5350005.00,religion_and_worship settlement_and_districts ...


In [12]:
# Rename the aggregated category text column to wiki_items_text
# so it matches the original Wikidata-token file schema.
cat_wikidata_tokens = cat_wikidata_tokens.rename(
    columns={"category": "wiki_items_text"}
)

# Keep the columns in the same order as all_wikidata_tokens.parquet.
cat_wikidata_tokens = cat_wikidata_tokens[
    ["loc_id", "attraction_count", "wiki_items_text"]
]

# Preview the final table before saving.
cat_wikidata_tokens.head()

,loc_id,attraction_count,wiki_items_text
0,5350001.00,13,parks_nature_and_recreation generic_buildings_...
1,5350002.00,23,transport_and_transit parks_nature_and_recreat...
2,5350003.00,7,generic_buildings_and_structures retail_and_co...
3,5350004.00,4,generic_buildings_and_structures culture_and_p...
4,5350005.00,10,religion_and_worship settlement_and_districts ...


In [13]:
# =========================================================
# 6. Save the categorised Wikidata token file
# =========================================================

# Save the final loc_id-level category-token table in the same graph folder.
output_path = graph_dir / "cat_all_wikidata_tokens.parquet"
cat_wikidata_tokens.to_parquet(output_path, index=False)

# Show where the file was saved and preview the saved data.
print(f"Saved to: {output_path}")
cat_wikidata_tokens.head()

Saved to: /home/najla/dev/najla-msc/bikeshare/data/processed/graph/cat_all_wikidata_tokens.parquet


,loc_id,attraction_count,wiki_items_text
0,5350001.00,13,parks_nature_and_recreation generic_buildings_...
1,5350002.00,23,transport_and_transit parks_nature_and_recreat...
2,5350003.00,7,generic_buildings_and_structures retail_and_co...
3,5350004.00,4,generic_buildings_and_structures culture_and_p...
4,5350005.00,10,religion_and_worship settlement_and_districts ...


graph only

In [16]:
# =========================================================
# 7. Add graph-only nodes with no Wikidata tokens
# =========================================================

# Read graph nodes.
# loc_id is saved as the index in nodes.parquet, so reset_index() brings it back as a column.
nodes_df = pd.read_parquet(graph_dir / "nodes.parquet").reset_index()

# Get every graph loc_id.
graph_loc_ids = nodes_df["loc_id"].drop_duplicates()

# Get loc_id values that already have categorised Wikidata tokens.
token_loc_ids = cat_wikidata_tokens["loc_id"].drop_duplicates()

# Find graph loc_id values missing from the Wikidata token table.
missing_loc_ids = graph_loc_ids[~graph_loc_ids.isin(token_loc_ids)]

# Build placeholder rows for graph nodes that have no Wikidata data.
missing_wikidata_rows = pd.DataFrame(
    {
        "loc_id": missing_loc_ids,
        "attraction_count": -1,
        "wiki_items_text": "no_wikidata",
        "attraction_missing_flag": 1,
    }
)

# Mark existing Wikidata rows as not missing.
cat_wikidata_tokens["attraction_missing_flag"] = 0

# Append missing graph-only loc_id rows back into the token table.
graph_cat_all_wikidata_tokens = pd.concat(
    [
        cat_wikidata_tokens,
        missing_wikidata_rows,
    ],
    ignore_index=True,
)

# Count distinct loc_id values after adding missing graph nodes.
distinct_loc_id_count = graph_cat_all_wikidata_tokens["loc_id"].nunique()
print(f"Distinct loc_id count: {distinct_loc_id_count}")


Distinct loc_id count: 1248


In [ ]:

# Save graph-ready categorised Wikidata token file.
output_path = graph_dir / "graph_cat_all_wikidata_tokens.parquet"
graph_cat_all_wikidata_tokens.to_parquet(output_path, index=False)

print(f"Saved to: {output_path}")
graph_cat_all_wikidata_tokens.head()

Saved to: /home/najla/dev/najla-msc/bikeshare/data/processed/graph/graph_cat_all_wikidata_tokens.parquet


,loc_id,attraction_count,wiki_items_text,attraction_missing_flag
0,5350001.00,13,parks_nature_and_recreation generic_buildings_...,0
1,5350002.00,23,transport_and_transit parks_nature_and_recreat...,0
2,5350003.00,7,generic_buildings_and_structures retail_and_co...,0
3,5350004.00,4,generic_buildings_and_structures culture_and_p...,0
4,5350005.00,10,religion_and_worship settlement_and_districts ...,0
